In [3]:
%load_ext autoreload
%autoreload 2
%reload_ext autoreload

import sys
sys.path.append("../../")

import nest_asyncio
nest_asyncio.apply()

import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "vscode"            

import matplotlib.pyplot as plt
import matplotlib.pylab as pylab
import matplotlib.dates as mdates
plt.style.use('seaborn-v0_8-dark')
params = {'legend.fontsize': 'x-large',
        'figure.figsize': (16, 9),
        'axes.labelsize': 'x-large',
        'axes.titlesize':'x-large',
        'xtick.labelsize':'x-large',
        'ytick.labelsize':'x-large'}
pylab.rcParams.update(params)

import pandas as pd
import numpy as np
import QuantLib as ql
import rateslib as rl

import datetime
import pytz
NY_tz = pytz.timezone("America/New_York") 
CHI_tz = pytz.timezone("America/Chicago") 
UTC_tz = pytz.timezone("UTC")

import warnings
warnings.filterwarnings(
    "ignore",
    category=UserWarning,
)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [14]:
df = pd.read_csv(r"C:\Users\chris\Downloads\CFTC_CUMULATIVE_FOREX_2026_07_09\CFTC_CUMULATIVE_FOREX_2026_07_09.csv")
df["Unique Product Identifier"].value_counts()
df["Exchange rate basis"].value_counts().head(50)

C:\Users\chris\AppData\Local\Temp\ipykernel_84568\3618460179.py:1: DtypeWarning:

Columns (35,102) have mixed types. Specify dtype option on import or set low_memory=False.



Exchange rate basis
USD/KRW    17426
USD/INR     9139
USD/TWD     8423
USD/BRL     4277
USD/IDR     3264
USD/PHP     2416
USD/CLP     2273
USD/COP     1726
KRW/USD      890
INR/USD      759
EUR/USD      722
USD/JPY      664
USD/CNY      604
TWD/USD      447
USD/PEN      368
AUD/USD      360
USD/MXN      332
GBP/USD      302
BRL/USD      292
IDR/USD      288
USD/CAD      277
COP/USD      243
PHP/USD      225
USD/EUR      224
USD/MYR      212
CLP/USD      203
CAD/USD      201
EUR/GBP      168
USD/ZAR      123
EUR/KRW      120
EUR/HUF      119
NZD/USD      119
USD/HKD      111
USD/CHF      106
USD/AUD      101
JPY/USD       97
CAD/BRL       92
EUR/AUD       92
AUD/NZD       91
EUR/BRL       90
CNY/USD       87
USD/VND       86
JPY/EUR       76
EUR/PLN       75
GBP/EUR       73
USD/KZT       68
EUR/CHF       67
EUR/CNY       66
USD/NZD       64
USD/EGP       64
Name: count, dtype: int64

In [12]:
cache_path = r"C:\Users\chris\clee\project-oasis\private\sdranalytics\.cache"

# as_of = datetime.date(2026, 2, 23)
# start = NY_tz.localize(datetime.datetime(as_of.year, as_of.month, as_of.day, 0, 0))
# end = NY_tz.localize(datetime.datetime(as_of.year, as_of.month, as_of.day, 23, 59))

start = NY_tz.localize(datetime.datetime(2026, 7, 10, 0, 0))
end = NY_tz.localize(datetime.datetime(2026, 7, 10, 23, 59))

# mdp = IRSwapsMDP(source="ERIS_EOD_LIVE-QL_BASIC")
# pricer = mdp.get_pricer(request=dict(curve_name="USD-SOFR-1D", timestamp=start.date()))

from SDRUtils.data.builder import SDRDataBuilder
sdr = SDRDataBuilder(cache_path=cache_path, show_tqdm=True)
df = sdr.grab_sdr_trades(
	start_timestamp=start,
	end_timestamp=end,
	agency="CFTC",
	asset_class="RATES",
)
df

CACHE HIT...: 100%|██████████| 1/1 [00:00<00:00, 13.13it/s]


,Dissemination Identifier,Original Dissemination Identifier,Action type,Event type,Event timestamp,Amendment indicator,Asset Class,Product name,Cleared,Mandatory clearing indicator,...,Package transaction price notation,Package transaction spread,Package transaction spread currency,Package transaction spread notation,Physical delivery location-Leg 1,Delivery Type,Unique Product Identifier,UPI FISN,UPI Underlier Name,file_date
0,4117894927000000601,,NEWT,TRAD,2026-07-10 04:00:08+00:00,None,IR,None,I,False,...,NaN,,,NaN,None,None,QZB883814F13,NA/Swap OIS INR,INR-MIBOR-OIS Compound,2026-07-10
1,4117894055000000101,,NEWT,TRAD,2026-07-10 04:00:25+00:00,None,IR,None,I,False,...,NaN,,,NaN,None,None,QZNSX4NZX325,NA/Swap Fxd Flt KRW,KRW-CD-KSDA-Bloomberg,2026-07-10
2,4117894056000000201,,NEWT,TRAD,2026-07-10 04:00:29+00:00,None,IR,None,I,False,...,NaN,,,NaN,None,None,QZB883814F13,NA/Swap OIS INR,INR-MIBOR-OIS Compound,2026-07-10
3,4117894057000000301,,NEWT,TRAD,2026-07-10 04:00:35+00:00,None,IR,None,I,False,...,NaN,,,NaN,None,None,QZB883814F13,NA/Swap OIS INR,INR-MIBOR-OIS Compound,2026-07-10
4,4117897702000000301,,NEWT,TRAD,2026-07-10 04:00:55+00:00,None,IR,None,N,False,...,NaN,,,NaN,None,None,QZR118Q9NWN9,NA/Swap Flt Flt JPY USD,JPY-TONA-OIS Compound vs USD-SOFR-OIS Compound,2026-07-10
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12374,4140171870000000401,4140171868000000201,CORR,,2026-07-10 22:03:37+00:00,None,IR,None,N,False,...,1.0,,,NaN,None,None,QZ43KHZW90D5,NA/O P Epn OIS USD,NA/Swap OIS USD,2026-07-10
12375,4140190149000000201,,NEWT,TRAD,2026-07-10 22:10:28+00:00,None,IR,None,N,False,...,1.0,,,NaN,None,None,QZ63QRQHX2ZT,NA/O Call Epn OIS USD,NA/Swap OIS USD,2026-07-10
12376,4140192131000000101,4140190148000000101,CORR,,2026-07-10 22:10:33+00:00,None,IR,None,N,False,...,1.0,,,NaN,None,None,QZ43KHZW90D5,NA/O P Epn OIS USD,NA/Swap OIS USD,2026-07-10
12377,4140618520000000201,4139526797000000101,CORR,,2026-07-10 22:40:40+00:00,None,IR,None,I,False,...,NaN,,,NaN,None,None,QZD55DZNSRR9,NA/Swap OIS BRL,BRL-CDI,2026-07-10


In [7]:
[col for col in list(df.columns) if "pay" in str(col).lower()]

['Other payment amount',
 'Fixed rate payment frequency period-Leg 1',
 'Floating rate payment frequency period-Leg 1',
 'Fixed rate payment frequency period-Leg 2',
 'Floating rate payment frequency period-Leg 2',
 'Fixed rate payment frequency period multiplier-Leg 1',
 'Floating rate payment frequency period multiplier-Leg 1',
 'Fixed rate payment frequency period multiplier-Leg 2',
 'Floating rate payment frequency period multiplier-Leg 2',
 'Other payment type',
 'Other payment currency']

In [1]:
import requests
res = requests.get("https://www.astorridge.com/trade-radar-rv-trades-in-europe-26th-nov-james-rice-astor-ridge/")
res.text

'<!DOCTYPE html>\n<html class="no-touch" lang="en-GB" xmlns="http://www.w3.org/1999/xhtml">\n<head>\n<meta http-equiv="Content-Type" content="text/html; charset=UTF-8">\n<meta name="viewport" content="width=device-width, initial-scale=1">\n<link rel="profile" href="http://gmpg.org/xfn/11">\n<link rel="pingback" href="https://www.astorridge.com/xmlrpc.php">\n<meta name=\'robots\' content=\'index, follow, max-image-preview:large, max-snippet:-1, max-video-preview:-1\' />\n\n\t<!-- This site is optimized with the Yoast SEO plugin v20.9 - https://yoast.com/wordpress/plugins/seo/ -->\n\t<title>Trade Radar - RV trades in Europe 26th Nov, James Rice @Astor Ridge - Astorridge</title>\n\t<link rel="canonical" href="https://www.astorridge.com/trade-radar-rv-trades-in-europe-26th-nov-james-rice-astor-ridge/" />\n\t<meta property="og:locale" content="en_GB" />\n\t<meta property="og:type" content="article" />\n\t<meta property="og:title" content="Trade Radar - RV trades in Europe 26th Nov, James Ri

In [21]:
# df["Event timestamp"] = df["Event timestamp"].astype(str)
# df["Execution Timestamp"] = df["Execution Timestamp"].astype(str)
# df.to_csv(r"C:\Users\chris\clee\ARBS\notebooks\sdr\april_fomc_dated_sdr_trades.csv", index=False)	

In [3]:
# df[(df["Effective Date"].dt.date == datetime.date(2026, 4, 28)) & (df["Expiration Date"].dt.date == datetime.date(2026, 6, 16))]

In [28]:
# df[(df["UPI Underlier Name"].str.contains("vs", case=False, na=False))]["UPI Underlier Name"].value_counts()

# df[df["UPI Underlier Name"] == "USD-Federal Funds-OIS Compound vs USD-SOFR-OIS Compound"]

df[df["Dissemination Identifier"] == "4135370792000000201"][
    [
        "Dissemination Identifier",
        "Original Dissemination Identifier",
        "Effective Date",
        "Expiration Date",
        "Cleared",
        "Event timestamp",
        "Execution Timestamp",
        'Notional amount-Leg 1',
        "Package transaction price",
		'Package transaction spread',
		'Exchange rate basis',
		'Spread-Leg 1',
		'Spread-Leg 2',
        'Fixed rate-Leg 1',
        'Fixed rate-Leg 2',
		'Price',
        'Other payment amount',
        'UPI Underlier Name'
    ]
].to_dict(orient="records")

[{'Dissemination Identifier': '4135370792000000201',
  'Original Dissemination Identifier': '',
  'Effective Date': Timestamp('2026-07-29 00:00:00'),
  'Expiration Date': Timestamp('2026-09-16 00:00:00'),
  'Cleared': 'I',
  'Event timestamp': Timestamp('2026-07-10 16:57:28+0000', tz='UTC'),
  'Execution Timestamp': Timestamp('2026-07-10 16:57:28+0000', tz='UTC'),
  'Notional amount-Leg 1': '7,400,000,000',
  'Package transaction price': '-48,419',
  'Package transaction spread': '',
  'Exchange rate basis': None,
  'Spread-Leg 1': nan,
  'Spread-Leg 2': nan,
  'Fixed rate-Leg 1': '0.037097',
  'Fixed rate-Leg 2': nan,
  'Price': None,
  'Other payment amount': '22998.45996',
  'UPI Underlier Name': 'USD-Federal Funds-OIS Compound'}]

In [26]:
df[df["Dissemination Identifier"] == "4136247796000000201"].to_dict(orient="records")

[{'Dissemination Identifier': '4136247796000000201',
  'Original Dissemination Identifier': '',
  'Action type': 'NEWT',
  'Event type': 'TRAD',
  'Event timestamp': Timestamp('2026-07-10 17:38:47+0000', tz='UTC'),
  'Amendment indicator': None,
  'Asset Class': 'IR',
  'Product name': None,
  'Cleared': 'I',
  'Mandatory clearing indicator': False,
  'Execution Timestamp': Timestamp('2026-07-10 17:38:47+0000', tz='UTC'),
  'Effective Date': Timestamp('2026-07-14 00:00:00'),
  'Expiration Date': Timestamp('2038-07-14 00:00:00'),
  'Maturity date of the underlier': None,
  'Non-standardized term indicator': False,
  'Platform identifier': 'TSEF',
  'Prime brokerage transaction indicator': False,
  'Block trade election indicator': False,
  'Large notional off-facility swap election indicator': None,
  'Notional amount-Leg 1': '250,000,000+',
  'Notional amount-Leg 2': '250,000,000+',
  'Notional currency-Leg 1': 'USD',
  'Notional currency-Leg 2': 'USD',
  'Notional quantity-Leg 1': Non

In [1]:
list(df.columns)

NameError: name 'df' is not defined

In [13]:
basis_ois_upis = pd.read_csv(r'C:\Users\chris\clee\ARBS\SDRUtils\anna_dsb_upis\Rates-Swap-Basis_OIS.csv')
basis_upis = pd.read_csv(r'C:\Users\chris\clee\ARBS\SDRUtils\anna_dsb_upis\Rates-Swap-Basis.csv')
basis_upis = pd.concat([basis_upis, basis_ois_upis], ignore_index=True)
basis_upis

,TemplateVersion,Header_AssetClass,Header_InstrumentType,Header_UseCase,Header_Level,Identifier_UPI,Identifier_Status,Identifier_StatusReason,Identifier_LastUpdateDateTime,Derived_ClassificationType,...,Attributes_UnderlyingInstrumentIndexTermUnit,Attributes_OptionType,Attributes_UnderlyingInstrumentUPI,Attributes_OptionExerciseStyle,Attributes_ValuationMethodorTrigger,Derived_ReturnorPayoutTrigger,Attributes_UnderlyingAssetType,Attributes_UnderlyingInstrumentISIN,Attributes_UnderlierCharacteristic,Attributes_ReturnorPayoutTrigger
0,1,Rates,Swap,Basis,UPI,QZ6HLS5Z9BWB,New,NaN,2023-10-15 12:01:46,SRACSP,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1,Rates,Swap,Basis,UPI,QZZQ9QG7WFKX,New,NaN,2023-10-15 12:01:47,SRACSP,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1,Rates,Swap,Basis,UPI,QZRRB068536C,New,NaN,2023-10-15 12:02:42,SRACSP,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1,Rates,Swap,Basis,UPI,QZ18K4WXLJFN,New,NaN,2023-10-15 12:05:18,SRACSP,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1,Rates,Swap,Basis,UPI,QZTSTSHDLF6J,New,NaN,2023-10-15 12:10:05,SRACSP,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11100,1,Rates,Swap,Basis_OIS,UPI,QZ8F34ZHQXRK,New,NaN,2025-12-30 05:14:54,SRHCSC,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
11101,1,Rates,Swap,Basis_OIS,UPI,QZ4Q4WRG1691,New,NaN,2026-01-01 01:44:25,SRHCSC,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
11102,1,Rates,Swap,Basis_OIS,UPI,QZBSC9X5GW5C,New,NaN,2026-01-01 05:06:10,SRHDSC,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
11103,1,Rates,Swap,Basis_OIS,UPI,QZ6WFCMC0H9G,New,NaN,2026-01-06 08:14:51,SRHCSP,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [18]:
df[(df["Unique Product Identifier"].isin(basis_upis["Identifier_UPI"])) & (df["UPI Underlier Name"].str.contains("USD"))][["UPI Underlier Name", "Effective Date", "Expiration Date", "Cleared", "Event timestamp", "Execution Timestamp"]]

,UPI Underlier Name,Effective Date,Expiration Date,Cleared,Event timestamp,Execution Timestamp
3455,USD-Federal Funds-OIS Compound vs USD-SOFR-OIS...,2026-05-26,2030-05-26,I,2026-05-21 08:34:33+00:00,2026-05-21 08:34:33+00:00
3500,USD-Federal Funds-OIS Compound vs USD-SOFR-OIS...,2026-05-26,2030-05-26,I,2026-05-21 08:37:00+00:00,2026-05-21 08:34:33+00:00
5017,USD-Federal Funds-OIS Compound vs USD-SOFR-OIS...,2026-05-26,2036-05-26,I,2026-05-21 09:31:20+00:00,2026-05-21 09:31:20+00:00
5035,USD-Federal Funds-OIS Compound vs USD-SOFR-OIS...,2026-05-26,2046-05-26,I,2026-05-21 09:32:07+00:00,2026-05-21 09:32:07+00:00
6638,USD-Federal Funds-OIS Compound vs USD-SOFR-OIS...,2026-05-26,2030-05-26,I,2026-05-21 11:14:19+00:00,2026-05-21 11:14:19+00:00
6667,USD-Federal Funds-OIS Compound vs USD-SOFR-OIS...,2026-05-26,2030-05-26,I,2026-05-21 11:15:47+00:00,2026-05-21 11:14:19+00:00
6737,USD-SOFR-OIS Compound vs USD-SOFR-OIS Compound,2026-05-28,2026-07-30,N,2026-05-21 11:20:00+00:00,2026-05-21 11:20:00+00:00
7110,USD-Federal Funds-OIS Compound vs USD-SOFR-OIS...,2027-05-26,2028-05-26,I,2026-05-21 11:41:53+00:00,2026-05-21 11:41:53+00:00
7169,USD-Federal Funds-OIS Compound vs USD-SOFR-OIS...,2026-05-26,2030-05-26,I,2026-05-21 11:45:09+00:00,2026-05-21 11:45:09+00:00
7196,USD-Federal Funds-OIS Compound vs USD-SOFR-OIS...,2026-05-26,2030-05-26,I,2026-05-21 11:45:55+00:00,2026-05-21 11:45:09+00:00


In [5]:
(3.812 - 3.882) - (3.843 - 3.812)

-0.10100000000000042